# Notebook 05: SHAP TreeExplainer on Random Forest

**Purpose:** Compute SHAP values for the Random Forest models on both datasets. Generate bar and beeswarm plots. Compare SHAP feature ranking with Gini importance.

**Outputs:** `results/shap_*.npy`, `results/shap_vs_gini_*.csv`, `figures/`

In [1]:
import os
os.chdir('/home/elious/research_projects/mdpi_sensors_2026')

import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 600, 'savefig.dpi': 600,
    'figure.figsize': (10, 6), 'font.family': 'sans-serif',
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10
})
sns.set_palette('colorblind')
os.makedirs('figures', exist_ok=True)

def extract_class1(sv):
    """Return class-1 SHAP matrix for any format (list or 3D array)."""
    if isinstance(sv, list):
        return sv[1]
    if sv.ndim == 3:
        return sv[:, :, 1]
    return sv

print('Ready.')

Ready.


In [2]:
ugr_test  = pd.read_csv('data/processed/ugr_test.csv')
cic_test  = pd.read_csv('data/processed/cic_test.csv')
ugr_train = pd.read_csv('data/processed/ugr_train.csv')

FEAT_UGR = [c for c in ugr_train.columns if c != 'Prediction']
FEAT_CIC = [c for c in cic_test.columns if c not in ['label', 'label_binary']]

X_te_ugr = ugr_test[FEAT_UGR]
X_te_cic = cic_test[FEAT_CIC]

rf_ugr = joblib.load('results/models/ugr_RandomForest.joblib')
rf_cic = joblib.load('results/models/cic_RandomForest.joblib')
print('Models and data loaded.')
print(f'UGR test: {X_te_ugr.shape}, CIC test: {X_te_cic.shape}')

Models and data loaded.
UGR test: (17972, 49), CIC test: (39995, 46)


## SHAP on UGRansome2024 (Random Forest)

In [3]:
import os
import numpy as np

npy_file = 'results/shap_values_ugr_rf.npy'
if os.path.exists(npy_file):
    print(f'Loading cached SHAP values from {npy_file}')
    shap_arr_ugr = np.load(npy_file, allow_pickle=True)
    explainer_ugr = shap.TreeExplainer(rf_ugr)  # needed for expected_value downstream
else:
    print('Computing SHAP values for UGRansome RF ...')
    explainer_ugr = shap.TreeExplainer(rf_ugr)
    shap_vals_ugr = explainer_ugr.shap_values(X_te_ugr)
    shap_arr_ugr = extract_class1(shap_vals_ugr)  # handles list or 3D array
    
    np.save('results/shap_values_ugr_rf.npy', shap_arr_ugr)
    print(f'UGR SHAP values shape: {shap_arr_ugr.shape}')


Loading cached SHAP values from results/shap_values_ugr_rf.npy


In [4]:
# SHAP bar plot - top 20 features
shap_mean_ugr = np.abs(shap_arr_ugr).mean(axis=0)
shap_df_ugr = pd.DataFrame({'feature': FEAT_UGR, 'shap_importance': shap_mean_ugr})
shap_df_ugr = shap_df_ugr.sort_values('shap_importance', ascending=False).reset_index(drop=True)
shap_df_ugr['shap_rank'] = range(1, len(shap_df_ugr) + 1)

top20_ugr = shap_df_ugr.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_ugr['feature'][::-1], top20_ugr['shap_importance'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('figures/shap_bar_rf_ugr.png', dpi=600, bbox_inches='tight')
plt.close()
print('UGR SHAP bar plot saved.')
print(top20_ugr[['feature','shap_importance']].head(10).to_string())

UGR SHAP bar plot saved.
             feature  shap_importance
0           Flag_APS         0.095319
1         Clusters_2         0.077677
2         Clusters_1         0.075832
3                USD         0.057513
4      Netflow_Bytes         0.047634
5          Flag_APSF         0.022285
6       Family_Globe         0.021835
7  Threats_Blacklist         0.020087
8         Flag_APRSF         0.018630
9               Port         0.018362


In [5]:
# Compare SHAP vs Gini importance
gini_ugr = rf_ugr.feature_importances_
gini_df_ugr = pd.DataFrame({'feature': FEAT_UGR, 'gini_importance': gini_ugr})
gini_df_ugr = gini_df_ugr.sort_values('gini_importance', ascending=False).reset_index(drop=True)
gini_df_ugr['gini_rank'] = range(1, len(gini_df_ugr) + 1)

# Merge SHAP and Gini rankings
compare_ugr = shap_df_ugr[['feature','shap_importance','shap_rank']].merge(
    gini_df_ugr[['feature','gini_importance','gini_rank']], on='feature')
compare_ugr.to_csv('results/shap_vs_gini_ugr.csv', index=False)
print('UGR SHAP vs Gini saved.')
print(compare_ugr.head(10)[['feature','shap_rank','gini_rank']].to_string())

UGR SHAP vs Gini saved.
             feature  shap_rank  gini_rank
0           Flag_APS          1          1
1         Clusters_2          2          5
2         Clusters_1          3          4
3                USD          4          2
4      Netflow_Bytes          5          3
5          Flag_APSF          6          9
6       Family_Globe          7          7
7  Threats_Blacklist          8         10
8         Flag_APRSF          9         11
9               Port         10          6


## SHAP on CICIoT2023 (Random Forest)

In [6]:
import os
import numpy as np

npy_file = 'results/shap_values_cic_rf.npy'
if os.path.exists(npy_file):
    print(f'Loading cached SHAP values from {npy_file}')
    shap_arr_cic = np.load(npy_file, allow_pickle=True)
    explainer_cic = shap.TreeExplainer(rf_cic)  # needed for expected_value downstream
else:
    print('Computing SHAP values for CICIoT RF ...')
    explainer_cic = shap.TreeExplainer(rf_cic)
    shap_vals_cic = explainer_cic.shap_values(X_te_cic)
    shap_arr_cic = extract_class1(shap_vals_cic)  # handles list or 3D array
    
    np.save('results/shap_values_cic_rf.npy', shap_arr_cic)
    print(f'CIC SHAP values shape: {shap_arr_cic.shape}')


Loading cached SHAP values from results/shap_values_cic_rf.npy


In [7]:
shap_mean_cic = np.abs(shap_arr_cic).mean(axis=0)
shap_df_cic = pd.DataFrame({'feature': FEAT_CIC, 'shap_importance': shap_mean_cic})
shap_df_cic = shap_df_cic.sort_values('shap_importance', ascending=False).reset_index(drop=True)
shap_df_cic['shap_rank'] = range(1, len(shap_df_cic) + 1)

top20_cic = shap_df_cic.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_cic['feature'][::-1], top20_cic['shap_importance'][::-1], color='coral')
ax.set_xlabel('Mean |SHAP value|')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('figures/shap_bar_rf_cic.png', dpi=600, bbox_inches='tight')
plt.close()
print('CIC SHAP bar plot saved.')
print(top20_cic[['feature','shap_importance']].head(10).to_string())

CIC SHAP bar plot saved.
         feature  shap_importance
0      rst_count         0.065082
1            IAT         0.061223
2      urg_count         0.052081
3  Header_Length         0.034331
4       Magnitue         0.029426
5  flow_duration         0.026853
6         Weight         0.024571
7            AVG         0.022845
8            Max         0.022478
9       Variance         0.021079


In [8]:
gini_cic = rf_cic.feature_importances_
gini_df_cic = pd.DataFrame({'feature': FEAT_CIC, 'gini_importance': gini_cic})
gini_df_cic = gini_df_cic.sort_values('gini_importance', ascending=False).reset_index(drop=True)
gini_df_cic['gini_rank'] = range(1, len(gini_df_cic) + 1)

compare_cic = shap_df_cic[['feature','shap_importance','shap_rank']].merge(
    gini_df_cic[['feature','gini_importance','gini_rank']], on='feature')
compare_cic.to_csv('results/shap_vs_gini_cic.csv', index=False)
print('CIC SHAP vs Gini saved.')
print(compare_cic.head(10)[['feature','shap_rank','gini_rank']].to_string())

# Highlight IAT ranking disagreement
iat_row = compare_cic[compare_cic['feature'] == 'IAT']
if not iat_row.empty:
    print(f'\nIAT: SHAP rank={iat_row["shap_rank"].values[0]}, Gini rank={iat_row["gini_rank"].values[0]}')

CIC SHAP vs Gini saved.
         feature  shap_rank  gini_rank
0      rst_count          1          1
1            IAT          2         14
2      urg_count          3          2
3  Header_Length          4         10
4       Magnitue          5          4
5  flow_duration          6          9
6         Weight          7         16
7            AVG          8          8
8            Max          9          6
9       Variance         10          3

IAT: SHAP rank=2, Gini rank=14


In [9]:
# Generate beeswarm plots using shap library
# UGR beeswarm
expl_obj_ugr = shap.Explanation(values=shap_arr_ugr,
                                 base_values=explainer_ugr.expected_value if not isinstance(explainer_ugr.expected_value, list) else explainer_ugr.expected_value[1],
                                 data=X_te_ugr.values,
                                 feature_names=FEAT_UGR)
fig = plt.figure(figsize=(12, 8))
shap.plots.beeswarm(expl_obj_ugr, max_display=20, show=False)
plt.tight_layout()
plt.savefig('figures/shap_beeswarm_rf_ugr.png', dpi=600, bbox_inches='tight')
plt.close()
print('UGR beeswarm saved.')

# CIC beeswarm
expl_obj_cic = shap.Explanation(values=shap_arr_cic,
                                 base_values=explainer_cic.expected_value if not isinstance(explainer_cic.expected_value, list) else explainer_cic.expected_value[1],
                                 data=X_te_cic.values,
                                 feature_names=FEAT_CIC)
fig = plt.figure(figsize=(12, 8))
shap.plots.beeswarm(expl_obj_cic, max_display=20, show=False)
plt.tight_layout()
plt.savefig('figures/shap_beeswarm_rf_cic.png', dpi=600, bbox_inches='tight')
plt.close()
print('CIC beeswarm saved.')

UGR beeswarm saved.


CIC beeswarm saved.


## Summary

**Purpose:** Explain Random Forest predictions using SHAP TreeExplainer.

**Method:** Applied shap.TreeExplainer to the trained RF models. SHAP values computed on the full test set. Ranked features by mean absolute SHAP value. Compared SHAP ranking to Gini (MDI) importance ranking.

**Key findings:** See shap_vs_gini_*.csv for rank comparisons. Note IAT rank by SHAP vs Gini for CICIoT (the expected disagreement from the dissertation).